# Online Retail Dataset — SQL Analysis

**Goal:** Load the cleaned data into a SQLite database and answer our business questions using SQL.

We use SQLite because it's serverless, file-based, and perfect for a portfolio project — no setup needed.

In [1]:
import pandas as pd
import sqlite3

# Load the cleaned data
df = pd.read_csv('../data/online_retail_cleaned.csv', parse_dates=['InvoiceDate'])
print('Shape:', df.shape)
df.head()

C:\Users\navya\AppData\Local\Temp\ipykernel_3808\1645087437.py:5: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/online_retail_cleaned.csv', parse_dates=['InvoiceDate'])


Shape: (524878, 15)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,HasCustomerID,Revenue,InvoiceYear,InvoiceMonth,InvoiceYearMonth,InvoiceDay,InvoiceHour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,True,15.30,2010,12,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,True,20.34,2010,12,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,True,22.00,2010,12,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,True,20.34,2010,12,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,True,20.34,2010,12,2010-12,Wednesday,8


In [2]:
# Create a SQLite database file and load the dataframe into a table called 'sales'
conn = sqlite3.connect('../data/retail.db')
df.to_sql('sales', conn, if_exists='replace', index=False)
print('Loaded into SQLite as table "sales"')

Loaded into SQLite as table "sales"


In [3]:
# Helper function so we can run SQL and get a clean dataframe back each time
def run_query(query):
    return pd.read_sql_query(query, conn)

## Q1: What is the total revenue, and how does it trend month-over-month?

In [4]:
query1 = """
SELECT 
    InvoiceYearMonth,
    ROUND(SUM(Revenue), 2) AS TotalRevenue,
    COUNT(DISTINCT InvoiceNo) AS NumOrders
FROM sales
GROUP BY InvoiceYearMonth
ORDER BY InvoiceYearMonth;
"""
monthly_revenue = run_query(query1)
monthly_revenue

,InvoiceYearMonth,TotalRevenue,NumOrders
0,2010-12,821452.73,1559
1,2011-01,689811.61,1086
2,2011-02,522545.56,1100
3,2011-03,716215.26,1454
4,2011-04,536968.49,1246
5,2011-05,769296.61,1681
6,2011-06,760547.01,1533
7,2011-07,718076.12,1475
8,2011-08,757841.38,1361
9,2011-09,1056435.19,1837


## Q2: Which products generate the highest revenue?

In [5]:
query2 = """
SELECT 
    Description,
    SUM(Quantity) AS TotalQuantitySold,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM sales
GROUP BY Description
ORDER BY TotalRevenue DESC
LIMIT 10;
"""
top_products = run_query(query2)
top_products

,Description,TotalQuantitySold,TotalRevenue
0,DOTCOM POSTAGE,706,206248.77
1,REGENCY CAKESTAND 3 TIER,13851,174156.54
2,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
3,WHITE HANGING HEART T-LIGHT HOLDER,37872,106236.72
4,PARTY BUNTING,18283,99445.23
5,JUMBO BAG RED RETROSPOT,48371,94159.81
6,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
7,POSTAGE,3150,78101.88
8,Manual,6985,77752.82
9,RABBIT NIGHT LIGHT,30739,66870.03


## Q3: Which countries generate the most revenue, and how concentrated is it?

In [6]:
query3 = """
SELECT 
    Country,
    ROUND(SUM(Revenue), 2) AS TotalRevenue,
    COUNT(DISTINCT InvoiceNo) AS NumOrders,
    ROUND(SUM(Revenue) * 100.0 / (SELECT SUM(Revenue) FROM sales), 2) AS PctOfTotalRevenue
FROM sales
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10;
"""
revenue_by_country = run_query(query3)
revenue_by_country

,Country,TotalRevenue,NumOrders,PctOfTotalRevenue
0,United Kingdom,9001744.09,18019,84.59
1,Netherlands,285446.34,94,2.68
2,EIRE,283140.52,288,2.66
3,Germany,228678.40,457,2.15
4,France,209625.37,392,1.97
5,Australia,138453.81,57,1.30
6,Spain,61558.56,90,0.58
7,Switzerland,57067.60,54,0.54
8,Belgium,41196.34,98,0.39
9,Sweden,38367.83,36,0.36


## Q4: Who are the top 10 customers by total spend?

Note: we filter out rows with missing CustomerID since we can't attribute them to a specific customer.

In [7]:
query4 = """
SELECT 
    CustomerID,
    Country,
    ROUND(SUM(Revenue), 2) AS TotalSpend,
    COUNT(DISTINCT InvoiceNo) AS NumOrders
FROM sales
WHERE HasCustomerID = 1
GROUP BY CustomerID
ORDER BY TotalSpend DESC
LIMIT 10;
"""
top_customers = run_query(query4)
top_customers

,CustomerID,Country,TotalSpend,NumOrders
0,14646.0,Netherlands,280206.02,73
1,18102.0,United Kingdom,259657.30,60
2,17450.0,United Kingdom,194390.79,46
3,16446.0,United Kingdom,168472.50,2
4,14911.0,EIRE,143711.17,201
5,12415.0,Australia,124914.53,21
6,14156.0,EIRE,117210.08,55
7,17511.0,United Kingdom,91062.38,31
8,16029.0,United Kingdom,80850.84,63
9,12346.0,United Kingdom,77183.60,1


## Q5: What's the average order value, and how does it vary by country?

In [8]:
query5 = """
WITH order_totals AS (
    SELECT 
        InvoiceNo,
        Country,
        SUM(Revenue) AS OrderRevenue
    FROM sales
    GROUP BY InvoiceNo, Country
)
SELECT 
    Country,
    ROUND(AVG(OrderRevenue), 2) AS AvgOrderValue,
    COUNT(*) AS NumOrders
FROM order_totals
GROUP BY Country
HAVING NumOrders >= 10
ORDER BY AvgOrderValue DESC
LIMIT 10;
"""
avg_order_value = run_query(query5)
avg_order_value

,Country,AvgOrderValue,NumOrders
0,Netherlands,3036.66,94
1,Australia,2429.01,57
2,Japan,1969.28,19
3,Hong Kong,1407.55,11
4,Sweden,1065.77,36
5,Switzerland,1056.81,54
6,Denmark,1053.07,18
7,Norway,1004.60,36
8,EIRE,983.13,288
9,Cyprus,843.93,16


## Q6: What's the distribution of orders per customer? (One-time vs repeat buyers)

In [9]:
query6 = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS NumOrders
    FROM sales
    WHERE HasCustomerID = 1
    GROUP BY CustomerID
)
SELECT 
    CASE 
        WHEN NumOrders = 1 THEN 'One-time buyer'
        WHEN NumOrders BETWEEN 2 AND 5 THEN '2-5 orders'
        WHEN NumOrders BETWEEN 6 AND 10 THEN '6-10 orders'
        ELSE '10+ orders'
    END AS CustomerType,
    COUNT(*) AS NumCustomers
FROM customer_orders
GROUP BY CustomerType
ORDER BY NumCustomers DESC;
"""
customer_distribution = run_query(query6)
customer_distribution

,CustomerType,NumCustomers
0,2-5 orders,1973
1,One-time buyer,1493
2,6-10 orders,535
3,10+ orders,337


## Q7: What day of week and hour do most orders happen?

In [10]:
query7 = """
SELECT 
    InvoiceDay,
    COUNT(DISTINCT InvoiceNo) AS NumOrders,
    ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM sales
GROUP BY InvoiceDay
ORDER BY NumOrders DESC;
"""
orders_by_day = run_query(query7)
orders_by_day

,InvoiceDay,NumOrders,TotalRevenue
0,Thursday,4246,2199292.57
1,Wednesday,3690,1847074.38
2,Tuesday,3554,2175700.51
3,Friday,3140,1837470.49
4,Monday,3126,1775782.07
5,Sunday,2204,806790.78


## Q8: Which customers haven't purchased recently? (Recency analysis)

We calculate days since each customer's last purchase, relative to the last date in the dataset.

In [11]:
query8 = """
WITH last_purchase AS (
    SELECT 
        CustomerID,
        MAX(InvoiceDate) AS LastPurchaseDate,
        ROUND(SUM(Revenue), 2) AS TotalSpend
    FROM sales
    WHERE HasCustomerID = 1
    GROUP BY CustomerID
)
SELECT 
    CustomerID,
    LastPurchaseDate,
    TotalSpend,
    CAST(JULIANDAY((SELECT MAX(InvoiceDate) FROM sales)) - JULIANDAY(LastPurchaseDate) AS INTEGER) AS DaysSinceLastPurchase
FROM last_purchase
ORDER BY DaysSinceLastPurchase DESC
LIMIT 10;
"""
recency = run_query(query8)
recency

,CustomerID,LastPurchaseDate,TotalSpend,DaysSinceLastPurchase
0,12791.0,2010-12-01 11:27:00,192.60,373
1,13747.0,2010-12-01 10:37:00,79.60,373
2,14729.0,2010-12-01 12:43:00,313.49,373
3,16583.0,2010-12-01 12:03:00,233.45,373
4,17908.0,2010-12-01 11:45:00,232.03,373
5,17968.0,2010-12-01 12:23:00,265.10,373
6,18074.0,2010-12-01 09:53:00,489.60,373
7,12855.0,2010-12-02 09:37:00,38.10,372
8,13065.0,2010-12-01 16:52:00,205.86,372
9,13108.0,2010-12-02 10:35:00,350.06,372


## Q9: Which products have the highest cancellation/return rates?

We load the cancellations file we saved separately during cleaning and join it against total sales per product.

In [12]:
cancellations = pd.read_csv('../data/cancellations.csv')
cancellations.to_sql('cancellations', conn, if_exists='replace', index=False)

query9 = """
SELECT 
    c.Description,
    COUNT(*) AS NumCancellations,
    SUM(ABS(c.Quantity)) AS TotalQuantityCancelled
FROM cancellations c
GROUP BY c.Description
ORDER BY NumCancellations DESC
LIMIT 10;
"""
top_cancellations = run_query(query9)
top_cancellations

,Description,NumCancellations,TotalQuantityCancelled
0,Manual,244,4066
1,REGENCY CAKESTAND 3 TIER,181,857
2,POSTAGE,126,147
3,JAM MAKING SET WITH JARS,87,247
4,Discount,77,1194
5,SET OF 3 CAKE TINS PANTRY DESIGN,74,157
6,SAMPLES,61,61
7,STRAWBERRY CERAMIC TRINKET BOX,55,363
8,ROSES REGENCY TEACUP AND SAUCER,54,437
9,RECIPE BOX PANTRY YELLOW DESIGN,47,151


## Export key results for use in Excel and write-up

In [13]:
monthly_revenue.to_csv('../data/results_monthly_revenue.csv', index=False)
top_products.to_csv('../data/results_top_products.csv', index=False)
revenue_by_country.to_csv('../data/results_revenue_by_country.csv', index=False)
top_customers.to_csv('../data/results_top_customers.csv', index=False)
avg_order_value.to_csv('../data/results_avg_order_value.csv', index=False)
customer_distribution.to_csv('../data/results_customer_distribution.csv', index=False)
orders_by_day.to_csv('../data/results_orders_by_day.csv', index=False)
recency.to_csv('../data/results_recency.csv', index=False)
top_cancellations.to_csv('../data/results_top_cancellations.csv', index=False)

print('All query results exported to /data as CSVs — ready for Excel dashboard.')

All query results exported to /data as CSVs — ready for Excel dashboard.


In [14]:
conn.close()